# FDA Drug Adverse Events: Installer

Installs the complete FDA Human Drug Adverse Event (FAERS) medallion sample into **the workspace this notebook runs in**, then starts the pipeline that downloads the data and builds the Silver and Gold tables. No manual steps are needed: you don't attach a lakehouse or import other files.

| Step | What it does |
|---|---|
| 1 | Downloads the notebooks and the pipeline definition from the [GitHub repository](https://github.com/isinghrana/fabric-samples-healthcare) |
| 2 | Creates a schema-enabled lakehouse, or reuses one with the same name |
| 3 | Installs `nb_fda_00_download_dataset`, `nb_fda_01_silver_load` and `nb_fda_02_gold_build` with the lakehouse as their default lakehouse |
| 4 | Installs the `pl_fda_one_time_build` pipeline: Download Dataset → Load Silver → Build Gold |
| 5 | Starts the pipeline and finishes **without waiting** for it. The pipeline keeps running on its own |

**Before you run it**
- The workspace must be on a Fabric capacity. F64 is recommended, because the Silver and Gold notebooks use 8 executors.
- You need the Contributor role or higher in the workspace.
- The full dataset needs about 700 GB of OneLake storage (~115 GB of zips plus ~580 GB of JSON) and takes several hours to process. To try the sample quickly, set `p_quarters` to a few quarters, for example `2025q1,2025q2`.

**Re-running is safe**: existing items are updated instead of duplicated, the lakehouse and its data are never deleted, and a second pipeline run is not started while one is still running.

> FAERS reports are unvalidated and cannot establish causality or incidence. See the disclaimer in the repository's `medallion-architecture-plan.md`.

## Session size

The installer only calls the Fabric REST API and GitHub; it does not run Spark jobs. The cell below keeps the Spark session as small as possible: one executor, which is the minimum a session can have. It uses the Starter Pool's Medium node size so the session starts quickly.

In [ ]:
%%configure
{
    "driverCores": 8,
    "driverMemory": "56g",
    "executorCores": 8,
    "executorMemory": "56g",
    "conf": {
        "spark.dynamicAllocation.minExecutors": "1",
        "spark.dynamicAllocation.maxExecutors": "1"
    }
}

## Parameters

The defaults install everything from the `main` branch and process the full dataset.

In [ ]:
p_lakehouse_name = "fda_adverse_events_lakehouse"           # lakehouse to create, or reuse if it already exists
p_github_repo = "isinghrana/fabric-samples-healthcare"      # GitHub repository holding the artifacts
p_github_branch = "main"                                    # branch to install from
p_github_folder = "fda_adverse_event_dataset"               # folder holding install-manifest.json and the artifacts
p_quarters = ""                                             # e.g. "2025q1,2025q2" for a small demo install; empty = full dataset
p_run_pipeline = True                                       # False installs the items without starting the pipeline

## Helpers

- **GitHub** files are read anonymously from `raw.githubusercontent.com`, because the repository is public.
- **Fabric REST API** calls use the signed-in user's token and the ID of the workspace this notebook runs in.
- Long-running operations (HTTP 202) are polled until they finish, and throttled calls (HTTP 429) are retried after the `Retry-After` delay.
- Any failed call stops the notebook and prints the HTTP status and error body.

In [ ]:
import base64
import json
import re
import time

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

FABRIC_API = "https://api.fabric.microsoft.com/v1"
FABRIC_PORTAL = "https://app.fabric.microsoft.com"
LRO_TIMEOUT_SECONDS = 600

WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]


def as_bool(value):
    return value if isinstance(value, bool) else str(value).strip().lower() in ("true", "1", "yes")


def param(value, default):
    if value is None or (isinstance(value, str) and value.strip() == ""):
        return default
    return value


LAKEHOUSE_NAME = str(param(p_lakehouse_name, "fda_adverse_events_lakehouse")).strip()
GITHUB_REPO = str(param(p_github_repo, "isinghrana/fabric-samples-healthcare")).strip()
GITHUB_BRANCH = str(param(p_github_branch, "main")).strip()
GITHUB_FOLDER = str(param(p_github_folder, "fda_adverse_event_dataset")).strip().strip("/")
QUARTERS = ",".join(q.strip().lower() for q in str(param(p_quarters, "")).split(",") if q.strip())
RUN_PIPELINE = as_bool(param(p_run_pipeline, True))

if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{0,255}", LAKEHOUSE_NAME):
    raise ValueError("p_lakehouse_name must start with a letter and contain only letters, numbers and underscores")

http = requests.Session()
http.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]), respect_retry_after_header=True)))


def fabric_headers():
    return {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}",
            "Content-Type": "application/json"}


def fabric(method, path, body=None, expected=(200, 201, 202)):
    """Call the Fabric REST API, retrying throttled calls and waiting for long-running operations."""
    url = path if path.startswith("https://") else f"{FABRIC_API}{path}"
    for attempt in range(6):
        response = http.request(method, url, headers=fabric_headers(),
                                data=json.dumps(body) if body is not None else None)
        if response.status_code != 429:
            break
        time.sleep(int(response.headers.get("Retry-After", 30)))
    if response.status_code not in expected:
        raise RuntimeError(f"{method} {url} failed: HTTP {response.status_code}\n{response.text[:2000]}")
    return response


def wait_for_operation(response):
    """Poll a long-running operation (HTTP 202) until it finishes."""
    if response.status_code != 202:
        return
    location = response.headers["Location"]
    deadline = time.time() + LRO_TIMEOUT_SECONDS
    while time.time() < deadline:
        time.sleep(int(response.headers.get("Retry-After", 5)))
        response = fabric("GET", location, expected=(200, 202))
        state = response.json().get("status") if response.content else None
        if state == "Succeeded":
            return
        if state in ("Failed", "Cancelled"):
            raise RuntimeError(f"Operation {state.lower()}: {response.text[:2000]}")
    raise TimeoutError(f"Operation did not finish within {LRO_TIMEOUT_SECONDS} seconds: {location}")


def list_items(path):
    """Return every item from a paged Fabric list API."""
    items, url = [], path
    while url:
        page = fabric("GET", url).json()
        items.extend(page.get("value", []))
        url = page.get("continuationUri")
    return items


def find_item(item_type, name):
    matches = [i for i in list_items(f"/workspaces/{WORKSPACE_ID}/items?type={item_type}") if i["displayName"] == name]
    return matches[0] if matches else None


def to_base64(text):
    return base64.b64encode(text.encode("utf-8")).decode("ascii")


def create_or_update(item_type, name, description, definition):
    """Create the item, or replace the definition of an existing item with the same name."""
    existing = find_item(item_type, name)
    if existing:
        wait_for_operation(fabric("POST", f"/workspaces/{WORKSPACE_ID}/items/{existing['id']}/updateDefinition",
                                  {"definition": definition}))
        return existing["id"], "updated"
    wait_for_operation(fabric("POST", f"/workspaces/{WORKSPACE_ID}/items",
                              {"displayName": name, "type": item_type, "description": description,
                               "definition": definition}))
    created = find_item(item_type, name)
    if not created:
        raise RuntimeError(f"{item_type} '{name}' was not found after it was created")
    return created["id"], "created"


installed = []   # (type, name, id, action) for the summary
print(f"Workspace: {WORKSPACE_ID}")
print(f"Lakehouse: {LAKEHOUSE_NAME}")
print(f"Source:    https://github.com/{GITHUB_REPO}/tree/{GITHUB_BRANCH}/{GITHUB_FOLDER}")
print(f"Quarters:  {QUARTERS or 'all (full dataset)'}")

## Step 1: Download the artifacts from GitHub

`install-manifest.json` lists the notebooks and pipeline to install. Every file is downloaded and checked to be valid JSON **before** anything is created in Fabric, so a wrong branch or a missing file stops the install without leaving a half-installed workspace.

In [ ]:
GITHUB_BASE = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}/{GITHUB_FOLDER}"


def github_json(file_name):
    url = f"{GITHUB_BASE}/{file_name}"
    response = http.get(url, timeout=60)
    if response.status_code != 200:
        raise RuntimeError(f"Could not download {url}: HTTP {response.status_code}. "
                           "Check p_github_repo, p_github_branch and p_github_folder.")
    try:
        return json.loads(response.text)
    except ValueError as error:
        raise RuntimeError(f"{url} is not valid JSON: {error}") from error


manifest = github_json("install-manifest.json")
notebook_sources = {nb["name"]: github_json(nb["file"]) for nb in manifest["notebooks"]}
pipeline_sources = {pl["name"]: github_json(pl["file"]) for pl in manifest["pipelines"]}

for entry in manifest["notebooks"] + manifest["pipelines"]:
    print(f"Downloaded {entry['file']}")

## Step 2: Create or reuse the lakehouse

The lakehouse is created with **schemas enabled**, because the notebooks write to the `silver` and `gold` schemas. If a lakehouse with the same name already exists, it is reused together with any data already in it. The install stops if that lakehouse doesn't have schemas enabled.

In [ ]:
def find_lakehouse(name):
    matches = [lh for lh in list_items(f"/workspaces/{WORKSPACE_ID}/lakehouses") if lh["displayName"] == name]
    return matches[0] if matches else None


lakehouse = find_lakehouse(LAKEHOUSE_NAME)
if lakehouse:
    lakehouse_action = "reused"
else:
    wait_for_operation(fabric("POST", f"/workspaces/{WORKSPACE_ID}/lakehouses", {
        "displayName": LAKEHOUSE_NAME,
        "description": "FDA Human Drug Adverse Event (FAERS) medallion: raw JSON files, Silver and Gold tables",
        "creationPayload": {"enableSchemas": True},
    }))
    lakehouse = find_lakehouse(LAKEHOUSE_NAME)
    lakehouse_action = "created"
    if not lakehouse:
        raise RuntimeError(f"Lakehouse '{LAKEHOUSE_NAME}' was not found after it was created")

LAKEHOUSE_ID = lakehouse["id"]
lakehouse = fabric("GET", f"/workspaces/{WORKSPACE_ID}/lakehouses/{LAKEHOUSE_ID}").json()
if not lakehouse.get("properties", {}).get("defaultSchema"):
    raise RuntimeError(f"Lakehouse '{LAKEHOUSE_NAME}' exists but does not have schemas enabled. "
                       "Choose another p_lakehouse_name.")

installed.append(("Lakehouse", LAKEHOUSE_NAME, LAKEHOUSE_ID, lakehouse_action))
print(f"Lakehouse {lakehouse_action}: {LAKEHOUSE_NAME} ({LAKEHOUSE_ID})")

## Step 3: Install the notebooks

Before each notebook is uploaded, its default lakehouse (stored in the notebook metadata under `dependencies.lakehouse`) is set to the lakehouse from step 2. That is what lets the notebooks read and write `Files/...`, `silver.*` and `gold.*` without anyone attaching a lakehouse by hand.

In [ ]:
notebook_ids = {}
for entry in manifest["notebooks"]:
    name = entry["name"]
    notebook = notebook_sources[name]
    notebook.setdefault("metadata", {}).setdefault("dependencies", {})["lakehouse"] = {
        "default_lakehouse": LAKEHOUSE_ID,
        "default_lakehouse_name": LAKEHOUSE_NAME,
        "default_lakehouse_workspace_id": WORKSPACE_ID,
        "known_lakehouses": [{"id": LAKEHOUSE_ID}],
    }
    definition = {"format": "ipynb", "parts": [{
        "path": "notebook-content.ipynb",
        "payload": to_base64(json.dumps(notebook)),
        "payloadType": "InlineBase64",
    }]}
    item_id, action = create_or_update("Notebook", name, entry.get("description", ""), definition)
    notebook_ids[name] = item_id
    installed.append(("Notebook", name, item_id, action))
    print(f"Notebook {action}: {name} ({item_id})")

## Step 4: Install the pipeline

The pipeline file in GitHub uses placeholders instead of IDs: `{{workspace_id}}` and `{{notebook:<name>}}`. They are replaced with this workspace's ID and the IDs of the notebooks installed in step 3. The install stops if any placeholder is left unresolved.

In [ ]:
PLACEHOLDER_RE = re.compile(r"\{\{([^{}]+)\}\}")


def resolve_placeholder(match):
    key = match.group(1).strip()
    if key == "workspace_id":
        return WORKSPACE_ID
    if key.startswith("notebook:") and key[len("notebook:"):] in notebook_ids:
        return notebook_ids[key[len("notebook:"):]]
    raise ValueError(f"Unresolved placeholder in pipeline definition: {{{{{key}}}}}")


pipeline_ids = {}
for entry in manifest["pipelines"]:
    name = entry["name"]
    content = PLACEHOLDER_RE.sub(resolve_placeholder, json.dumps(pipeline_sources[name]))
    definition = {"parts": [{
        "path": "pipeline-content.json",
        "payload": to_base64(content),
        "payloadType": "InlineBase64",
    }]}
    item_id, action = create_or_update("DataPipeline", name, entry.get("description", ""), definition)
    pipeline_ids[name] = item_id
    installed.append(("Pipeline", name, item_id, action))
    print(f"Pipeline {action}: {name} ({item_id})")

## Step 5: Start the pipeline (without waiting)

The pipeline is started and the installer moves straight on; it does not wait for the pipeline to finish. If a run of the pipeline is already queued or in progress (for example when the installer is run a second time), that run is reported instead of starting another one.

In [ ]:
PIPELINE_NAME = manifest["runPipeline"]
PIPELINE_ID = pipeline_ids[PIPELINE_NAME]
run_id, run_status = None, "not started (p_run_pipeline = False)"

if RUN_PIPELINE:
    runs = list_items(f"/workspaces/{WORKSPACE_ID}/items/{PIPELINE_ID}/jobs/instances")
    active = [r for r in runs if r.get("status") in ("NotStarted", "InProgress")]
    if active:
        run_id, run_status = active[0]["id"], f"already {active[0]['status']}; no new run started"
    else:
        response = fabric("POST", f"/workspaces/{WORKSPACE_ID}/items/{PIPELINE_ID}/jobs/instances?jobType=Pipeline",
                          {"executionData": {"parameters": {"p_quarters": QUARTERS}}}, expected=(202,))
        run_id = response.headers.get("Location", "").rstrip("/").split("/")[-1] or None
        run_status = "started"

print(f"Pipeline {PIPELINE_NAME}: {run_status}" + (f" (run ID {run_id})" if run_id else ""))

## Summary

Lists everything that was installed and how to follow the pipeline run.

In [ ]:
item_paths = {"Lakehouse": "lakehouses", "Notebook": "synapsenotebooks", "Pipeline": "pipelines"}
rows = "".join(
    f"<tr><td>{item_type}</td><td><a href='{FABRIC_PORTAL}/groups/{WORKSPACE_ID}/{item_paths[item_type]}/{item_id}' "
    f"target='_blank'>{name}</a></td><td>{action}</td><td><code>{item_id}</code></td></tr>"
    for item_type, name, item_id, action in installed)
pipeline_url = f"{FABRIC_PORTAL}/groups/{WORKSPACE_ID}/pipelines/{PIPELINE_ID}"

if run_id:
    next_steps = (
        f"<p><b>The pipeline <code>{PIPELINE_NAME}</code> is running</b> (run ID <code>{run_id}</code>). "
        "It runs on its own; this notebook does not wait for it.</p>"
        "<ul>"
        f"<li>Check its status: open the <a href='{pipeline_url}' target='_blank'>pipeline</a> and select "
        f"<b>Run history</b>, or use the <a href='{FABRIC_PORTAL}/monitoringhub' target='_blank'>Monitoring hub</a>.</li>"
        "<li>It runs <b>Download Dataset</b> → <b>Load Silver</b> → <b>Build Gold</b>. The full dataset takes several "
        "hours; a few quarters take much less.</li>"
        "<li>If a step fails, rerun the pipeline. Files already downloaded and quarters already loaded are skipped.</li>"
        "</ul>")
else:
    next_steps = (f"<p>The pipeline was not started. Run <a href='{pipeline_url}' target='_blank'>{PIPELINE_NAME}</a> "
                  "when you are ready.</p>")

displayHTML(
    "<h3>FDA adverse events sample installed</h3>"
    "<table><tr><th>Type</th><th>Name</th><th>Action</th><th>ID</th></tr>" + rows + "</table>" + next_steps)

for item_type, name, item_id, action in installed:
    print(f"{item_type:<9} {action:<8} {name}  ({item_id})")
print(f"\nPipeline run: {run_status}" + (f", run ID {run_id}" if run_id else ""))
print(f"Check the status: {pipeline_url}  (Run history) or {FABRIC_PORTAL}/monitoringhub")